In [1]:
# %pip install pyannote.audio

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
access_token = os.getenv('HUGGINGFACE_ACCESS_TOKENS')

In [3]:
# instantiate the pipeline
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(  # HuggingFace 모델 download
  "pyannote/speaker-diarization-3.1",
  use_auth_token=access_token
  )

c:\workspaces\ai_agent\src_chap05\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [disable_jit_profiling, allow_tf32]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
c:\workspaces\ai_agent\src_chap05\.venv\Lib\site-packages\lightning_fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be fl

In [4]:
# run the pipeline on an audio file
# diarization = pipeline("audio.wav")
diarization = pipeline("../audio/싼기타_비싼기타.mp3")

# dump the diarization output to disk using RTTM format
with open("../audio/싼기타_비싼기타.rttm", "w", encoding='utf-8') as rttm:   # 데이터 저장
    diarization.write_rttm(rttm)

c:\workspaces\ai_agent\src_chap05\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
c:\workspaces\ai_agent\src_chap05\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
c:\workspaces\ai_agent\src_chap05\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits

In [5]:
# RTTM을 CSV로 변환
import pandas as pd

rttm_path = "../audio/싼기타_비싼기타.rttm"

df_rttm = pd.read_csv(
    rttm_path, # rttm 파일 경로
    sep=' ', # 구분자는 띄어쓰기
    header=None, # 헤더는 없음
    names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4'] 
)

display(df_rttm)


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,싼기타_비싼기타,1,414.481,2.970,NaN,NaN,SPEAKER_00,NaN,NaN
84,SPEAKER,싼기타_비싼기타,1,417.755,3.476,NaN,NaN,SPEAKER_01,NaN,NaN
85,SPEAKER,싼기타_비싼기타,1,423.644,0.776,NaN,NaN,SPEAKER_00,NaN,NaN
86,SPEAKER,싼기타_비싼기타,1,424.741,3.527,NaN,NaN,SPEAKER_00,NaN,NaN


In [6]:
# 발언이 끝난 시간 추가하기

df_rttm['end'] = df_rttm['start'] + df_rttm['duration']
display(df_rttm)


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204
...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,싼기타_비싼기타,1,414.481,2.970,NaN,NaN,SPEAKER_00,NaN,NaN,417.451
84,SPEAKER,싼기타_비싼기타,1,417.755,3.476,NaN,NaN,SPEAKER_01,NaN,NaN,421.231
85,SPEAKER,싼기타_비싼기타,1,423.644,0.776,NaN,NaN,SPEAKER_00,NaN,NaN,424.420
86,SPEAKER,싼기타_비싼기타,1,424.741,3.527,NaN,NaN,SPEAKER_00,NaN,NaN,428.268


In [7]:
# 판다스를 활용해 데이터프레임 형태로 저장하기
# 연속된 발화를 기록하기 위해 number 변수 추가하기

df_rttm['number'] = None
df_rttm.at[0, 'number'] = 0

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,None
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,None
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,None
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,None
...,...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,싼기타_비싼기타,1,414.481,2.970,NaN,NaN,SPEAKER_00,NaN,NaN,417.451,None
84,SPEAKER,싼기타_비싼기타,1,417.755,3.476,NaN,NaN,SPEAKER_01,NaN,NaN,421.231,None
85,SPEAKER,싼기타_비싼기타,1,423.644,0.776,NaN,NaN,SPEAKER_00,NaN,NaN,424.420,None
86,SPEAKER,싼기타_비싼기타,1,424.741,3.527,NaN,NaN,SPEAKER_00,NaN,NaN,428.268,None


In [8]:
# 화자 번호 매기기

for i in range(1, len(df_rttm)):
    if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"] + 1
    else:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]
    display(df_rttm.head(10)) 


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,None
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,None
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,None
5,SPEAKER,싼기타_비싼기타,1,32.414,0.759,NaN,NaN,SPEAKER_00,NaN,NaN,33.173,None
6,SPEAKER,싼기타_비싼기타,1,33.545,3.561,NaN,NaN,SPEAKER_00,NaN,NaN,37.106,None
7,SPEAKER,싼기타_비싼기타,1,37.628,3.763,NaN,NaN,SPEAKER_00,NaN,NaN,41.391,None
8,SPEAKER,싼기타_비싼기타,1,41.611,1.097,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,None
9,SPEAKER,싼기타_비싼기타,1,41.645,0.810,NaN,NaN,SPEAKER_01,NaN,NaN,42.455,None


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,0
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,None
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,None
5,SPEAKER,싼기타_비싼기타,1,32.414,0.759,NaN,NaN,SPEAKER_00,NaN,NaN,33.173,None
6,SPEAKER,싼기타_비싼기타,1,33.545,3.561,NaN,NaN,SPEAKER_00,NaN,NaN,37.106,None
7,SPEAKER,싼기타_비싼기타,1,37.628,3.763,NaN,NaN,SPEAKER_00,NaN,NaN,41.391,None
8,SPEAKER,싼기타_비싼기타,1,41.611,1.097,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,None
9,SPEAKER,싼기타_비싼기타,1,41.645,0.810,NaN,NaN,SPEAKER_01,NaN,NaN,42.455,None


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,0
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,0
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,None
5,SPEAKER,싼기타_비싼기타,1,32.414,0.759,NaN,NaN,SPEAKER_00,NaN,NaN,33.173,None
6,SPEAKER,싼기타_비싼기타,1,33.545,3.561,NaN,NaN,SPEAKER_00,NaN,NaN,37.106,None
7,SPEAKER,싼기타_비싼기타,1,37.628,3.763,NaN,NaN,SPEAKER_00,NaN,NaN,41.391,None
8,SPEAKER,싼기타_비싼기타,1,41.611,1.097,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,None
9,SPEAKER,싼기타_비싼기타,1,41.645,0.810,NaN,NaN,SPEAKER_01,NaN,NaN,42.455,None


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,0
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,0
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,0
5,SPEAKER,싼기타_비싼기타,1,32.414,0.759,NaN,NaN,SPEAKER_00,NaN,NaN,33.173,None
6,SPEAKER,싼기타_비싼기타,1,33.545,3.561,NaN,NaN,SPEAKER_00,NaN,NaN,37.106,None
7,SPEAKER,싼기타_비싼기타,1,37.628,3.763,NaN,NaN,SPEAKER_00,NaN,NaN,41.391,None
8,SPEAKER,싼기타_비싼기타,1,41.611,1.097,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,None
9,SPEAKER,싼기타_비싼기타,1,41.645,0.810,NaN,NaN,SPEAKER_01,NaN,NaN,42.455,None


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,0
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,0
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,0
5,SPEAKER,싼기타_비싼기타,1,32.414,0.759,NaN,NaN,SPEAKER_00,NaN,NaN,33.173,1
6,SPEAKER,싼기타_비싼기타,1,33.545,3.561,NaN,NaN,SPEAKER_00,NaN,NaN,37.106,None
7,SPEAKER,싼기타_비싼기타,1,37.628,3.763,NaN,NaN,SPEAKER_00,NaN,NaN,41.391,None
8,SPEAKER,싼기타_비싼기타,1,41.611,1.097,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,None
9,SPEAKER,싼기타_비싼기타,1,41.645,0.810,NaN,NaN,SPEAKER_01,NaN,NaN,42.455,None


...

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,0
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,0
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,0
5,SPEAKER,싼기타_비싼기타,1,32.414,0.759,NaN,NaN,SPEAKER_00,NaN,NaN,33.173,1
6,SPEAKER,싼기타_비싼기타,1,33.545,3.561,NaN,NaN,SPEAKER_00,NaN,NaN,37.106,1
7,SPEAKER,싼기타_비싼기타,1,37.628,3.763,NaN,NaN,SPEAKER_00,NaN,NaN,41.391,1
8,SPEAKER,싼기타_비싼기타,1,41.611,1.097,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,1
9,SPEAKER,싼기타_비싼기타,1,41.645,0.810,NaN,NaN,SPEAKER_01,NaN,NaN,42.455,2


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,0
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,0
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,0
5,SPEAKER,싼기타_비싼기타,1,32.414,0.759,NaN,NaN,SPEAKER_00,NaN,NaN,33.173,1
6,SPEAKER,싼기타_비싼기타,1,33.545,3.561,NaN,NaN,SPEAKER_00,NaN,NaN,37.106,1
7,SPEAKER,싼기타_비싼기타,1,37.628,3.763,NaN,NaN,SPEAKER_00,NaN,NaN,41.391,1
8,SPEAKER,싼기타_비싼기타,1,41.611,1.097,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,1
9,SPEAKER,싼기타_비싼기타,1,41.645,0.810,NaN,NaN,SPEAKER_01,NaN,NaN,42.455,2


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,0
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,0
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,0
5,SPEAKER,싼기타_비싼기타,1,32.414,0.759,NaN,NaN,SPEAKER_00,NaN,NaN,33.173,1
6,SPEAKER,싼기타_비싼기타,1,33.545,3.561,NaN,NaN,SPEAKER_00,NaN,NaN,37.106,1
7,SPEAKER,싼기타_비싼기타,1,37.628,3.763,NaN,NaN,SPEAKER_00,NaN,NaN,41.391,1
8,SPEAKER,싼기타_비싼기타,1,41.611,1.097,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,1
9,SPEAKER,싼기타_비싼기타,1,41.645,0.810,NaN,NaN,SPEAKER_01,NaN,NaN,42.455,2


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,0
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,0
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,0
5,SPEAKER,싼기타_비싼기타,1,32.414,0.759,NaN,NaN,SPEAKER_00,NaN,NaN,33.173,1
6,SPEAKER,싼기타_비싼기타,1,33.545,3.561,NaN,NaN,SPEAKER_00,NaN,NaN,37.106,1
7,SPEAKER,싼기타_비싼기타,1,37.628,3.763,NaN,NaN,SPEAKER_00,NaN,NaN,41.391,1
8,SPEAKER,싼기타_비싼기타,1,41.611,1.097,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,1
9,SPEAKER,싼기타_비싼기타,1,41.645,0.810,NaN,NaN,SPEAKER_01,NaN,NaN,42.455,2


,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.983,NaN,NaN,SPEAKER_01,NaN,NaN,11.388,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.927,NaN,NaN,SPEAKER_01,NaN,NaN,16.686,0
3,SPEAKER,싼기타_비싼기타,1,17.210,10.665,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,0
4,SPEAKER,싼기타_비싼기타,1,28.668,1.536,NaN,NaN,SPEAKER_01,NaN,NaN,30.204,0
5,SPEAKER,싼기타_비싼기타,1,32.414,0.759,NaN,NaN,SPEAKER_00,NaN,NaN,33.173,1
6,SPEAKER,싼기타_비싼기타,1,33.545,3.561,NaN,NaN,SPEAKER_00,NaN,NaN,37.106,1
7,SPEAKER,싼기타_비싼기타,1,37.628,3.763,NaN,NaN,SPEAKER_00,NaN,NaN,41.391,1
8,SPEAKER,싼기타_비싼기타,1,41.611,1.097,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,1
9,SPEAKER,싼기타_비싼기타,1,41.645,0.810,NaN,NaN,SPEAKER_01,NaN,NaN,42.455,2


In [9]:
# 같은 화자끼리 묶어서 정리하기
df_rttm_grouped = df_rttm.groupby('number').agg(    # aggrigation 집계 함수
    start=pd.NamedAgg(column='start', aggfunc='min'),  # grouping 안의 시작값
    end=pd.NamedAgg(column='end', aggfunc='max'), # grouping 안의 종료값
    speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
)

display(df_rttm_grouped)

,start,end,speaker_id
number,,,
0,0.993,30.204,SPEAKER_01
1,32.414,42.708,SPEAKER_00
2,41.645,44.024,SPEAKER_01
3,45.813,67.109,SPEAKER_00
4,67.227,82.786,SPEAKER_01
5,84.659,102.564,SPEAKER_00
6,103.492,117.532,SPEAKER_01
7,119.759,138.676,SPEAKER_00
8,139.351,168.967,SPEAKER_01


In [10]:
# 판다스를 활용해 데이터프레임 형태로 저장하기
# 발화 시간 추가하고 인덱스 제거하기

df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
df_rttm_grouped = df_rttm_grouped.reset_index(drop=True)
display(df_rttm_grouped)

,start,end,speaker_id,duration
0,0.993,30.204,SPEAKER_01,29.211
1,32.414,42.708,SPEAKER_00,10.294
2,41.645,44.024,SPEAKER_01,2.379
3,45.813,67.109,SPEAKER_00,21.296
4,67.227,82.786,SPEAKER_01,15.559
5,84.659,102.564,SPEAKER_00,17.905
6,103.492,117.532,SPEAKER_01,14.040
7,119.759,138.676,SPEAKER_00,18.917
8,139.351,168.967,SPEAKER_01,29.616
9,170.907,192.321,SPEAKER_00,21.414


In [11]:
# 판다스를 활용해 데이터프레임 형태로 저장하기
# 화자 분리 결과를 CSV 파일로 저장하기

df_rttm_grouped.to_csv(
    "../audio/싼기타_비싼기타_rttm.csv",
    sep=',',  # , 로 구분 지어서 저장
    index=False
)
